# Rozdział 2 — Bandici wieloręcy (wersja pełna)

Lektura: Sutton & Barto, rozdz. 2.

Implementujemy klasyczny testbed $k$-ramienny i porównujemy strategie eksploracji:
- $\varepsilon$-greedy
- UCB
- (opcjonalnie) gradient bandits


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline


## 1. Testbed $k$-ramiennego bandyty

Zaimplementuj symulator:
- $k$ akcji (ramion)
- każde ramię ma nieznany rozkład nagrody
- w każdym kroku wybierasz ramię i obserwujesz nagrodę


In [ ]:
from dataclasses import dataclass

@dataclass
class BanditConfig:
    k: int = 10
    reward_std: float = 1.0
    nonstationary: bool = False
    rw_std: float = 0.01  # random-walk std for q* if nonstationary

class KArmedBandit:
    def __init__(self, cfg: BanditConfig, seed: int = 0):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        self.q_star = self.rng.normal(loc=0.0, scale=1.0, size=cfg.k)

    def step(self, a: int) -> float:
        """Pull arm a and return a reward."""
        mean = self.q_star[a]
        r = self.rng.normal(loc=mean, scale=self.cfg.reward_std)
        if self.cfg.nonstationary:
            self.q_star += self.rng.normal(loc=0.0, scale=self.cfg.rw_std, size=self.cfg.k)
        return float(r)

    def optimal_action(self) -> int:
        return int(np.argmax(self.q_star))


## 2. Estymaty wartości akcji i aktualizacje przyrostowe

Trzymaj estymaty $Q(a)$ dla każdego ramienia.

Dwie popularne aktualizacje:

Średnia z próbek (krok $1/N(a)$):
$$Q_{t+1}(a) = Q_t(a) + \frac{1}{N_t(a)}\bigl(R_t - Q_t(a)\bigr)$$

Stały krok $\alpha$:
$$Q_{t+1}(a) = Q_t(a) + \alpha\bigl(R_t - Q_t(a)\bigr)$$


## Ćwiczenie 1 — implementacja aktualizacji przyrostowej

Zaimplementuj `update_estimate(Q, N, a, r, alpha=None)`:

- zaktualizuj licznik `N[a]`
- zaktualizuj `Q[a]` używając:
  - średniej z próbek, jeśli `alpha is None`
  - stałego kroku, jeśli podano `alpha`


> **Notatki dla prowadzącego:**  
> - Aktualizacja przyrostowa średniej: $Q \leftarrow Q + \alpha (r - Q)$.  
> - Gdy `alpha=None`, używamy $\alpha = 1/N[a]$ (średnia z próbek).  
> - Klucz: `N[a]` trzeba inkrementować **przed** policzeniem kroku `1/N[a]`.  
> - Test w notebooku to mini-unit-test: po dwóch identycznych nagrodach ma wyjść dokładnie 2.0.

In [ ]:
def update_q(Q: np.ndarray, N: np.ndarray, a: int, r: float, alpha: float = None):
    """Update Q[a] given reward r. N[a] has already been incremented."""
    if alpha is None:
        step = 1.0 / N[a]
    else:
        step = float(alpha)
    Q[a] += step * (r - Q[a])

# quick test: with alpha=None and two identical rewards, Q should equal that reward
Q = np.zeros(1)
N = np.zeros(1, dtype=int)
N[0] += 1
update_q(Q, N, 0, 2.0, alpha=None)
N[0] += 1
update_q(Q, N, 0, 2.0, alpha=None)
assert abs(Q[0] - 2.0) < 1e-9
print("Exercise 1 test passed ✅")


## 3. Wybór akcji $\varepsilon$-greedy

W każdym kroku:
- z prawdopodobieństwem $\varepsilon$: losowa akcja
- w przeciwnym razie: `argmax_a Q(a)`


## Ćwiczenie 2 — $\varepsilon$-greedy

Zaimplementuj `epsilon_greedy_action(Q, eps, rng)`.


> **Notatki dla prowadzącego:**  
> - $\varepsilon$-greedy: z prawdopodobieństwem $\varepsilon$ losuj akcję, wpp wybierz `argmax(Q)`.  
> - Remisy: `np.argmax` bierze pierwszy maks — można to zaakceptować albo losować wśród maksów.  
> - Dobre pytanie do grupy: co się stanie, jeśli ustawimy `eps=0` od początku?

In [ ]:
def epsilon_greedy_action(Q: np.ndarray, eps: float, rng: np.random.Generator) -> int:
    if rng.random() < eps:
        return int(rng.integers(0, len(Q)))
    return int(np.argmax(Q))

# sanity check: if eps=0, always greedy
rng = np.random.default_rng(0)
Q = np.array([0.0, 1.0, 0.5])
a = epsilon_greedy_action(Q, eps=0.0, rng=rng)
assert a == 1
print("Exercise 2 sanity check passed ✅")


## 4. UCB (Upper Confidence Bound)

UCB wybiera:

$$A_t = \arg\max_a \left[ Q_t(a) + c \sqrt{\frac{\log(t+1)}{N_t(a)+1e-8}} \right]$$

gdzie $c>0$ steruje poziomem eksploracji.


## Ćwiczenie 3 — UCB

Zaimplementuj `ucb_action(Q, N, t, c)`.


> **Notatki dla prowadzącego:**  
> - UCB1: $a_t = \arg\max_a\left[Q(a) + c\sqrt{\frac{\ln t}{N(a)}}\right]$.  
> - Obsłuż $N(a)=0$ (np. bonus $=\infty$), aby każda akcja była spróbowana.  
> - `t` powinno startować od 1 (żeby $\ln t$ było zdefiniowane).  
> - `c` steruje eksploracją: większe `c` = bardziej „ciekawski” agent.

In [ ]:
def ucb_action(Q: np.ndarray, N: np.ndarray, t: int, c: float) -> int:
    bonus = c * np.sqrt(np.log(t + 1.0) / (N + 1e-8))
    return int(np.argmax(Q + bonus))


## 5. Uruchom mały testbed

Wykonaj wiele uruchomień i narysuj:
- średnią nagrodę w czasie
- % wyboru akcji optymalnej

Porównaj $\varepsilon$-greedy vs UCB.


In [ ]:
def run_bandit(strategy: str, *, k: int = 10, steps: int = 1000, runs: int = 200, seed: int = 0):
    rng = np.random.default_rng(seed)
    avg_rewards = np.zeros(steps)
    pct_opt = np.zeros(steps)

    for run in range(runs):
        bandit = KArmedBandit(BanditConfig(k=k, nonstationary=False), seed=int(rng.integers(0, 1_000_000)))
        Q = np.zeros(k)
        N = np.zeros(k, dtype=int)

        for t in range(steps):
            if strategy == "egreedy":
                a = epsilon_greedy_action(Q, eps=0.1, rng=rng)
            elif strategy == "ucb":
                a = ucb_action(Q, N, t=t+1, c=2.0)
            else:
                raise ValueError("Unknown strategy")

            r = bandit.step(a)
            N[a] += 1
            update_q(Q, N, a, r, alpha=None)

            avg_rewards[t] += r
            pct_opt[t] += int(a == bandit.optimal_action())

    return {"avg_rewards": avg_rewards / runs, "pct_opt": 100 * (pct_opt / runs)}

out_eps = run_bandit("egreedy")
out_ucb = run_bandit("ucb")


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(out_eps["avg_rewards"], label="ε-greedy (ε=0.1)")
plt.plot(out_ucb["avg_rewards"], label="UCB (c=2)")
plt.xlabel("Steps")
plt.ylabel("Average reward")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10,4))
plt.plot(out_eps["pct_opt"], label="ε-greedy (ε=0.1)")
plt.plot(out_ucb["pct_opt"], label="UCB (c=2)")
plt.xlabel("Steps")
plt.ylabel("% optimal action")
plt.legend()
plt.grid(True)
plt.show()


## (Opcjonalnie) 6. Gradient bandits

Jeśli starczy czasu:
- utrzymuj preferencje $H(a)$
- użyj polityki softmax $$\pi(a) = \frac{e^{H(a)}}{\sum_b e^{H(b)}}$$
- aktualizuj preferencje regułą typu REINFORCE (zob. Sutton & Barto)


In [ ]:
def softmax(x: np.ndarray) -> np.ndarray:
    x = x - np.max(x)
    ex = np.exp(x)
    return ex / np.sum(ex)

def run_gradient_bandit(k: int = 10, steps: int = 1000, runs: int = 200, alpha: float = 0.1, seed: int = 0):
    rng = np.random.default_rng(seed)
    avg_rewards = np.zeros(steps)

    for run in range(runs):
        bandit = KArmedBandit(BanditConfig(k=k, nonstationary=False), seed=int(rng.integers(0, 1_000_000)))
        H = np.zeros(k)
        pi = softmax(H)
        baseline = 0.0

        for t in range(steps):
            pi = softmax(H)
            a = int(rng.choice(k, p=pi))
            r = bandit.step(a)

            # running-average baseline
            baseline += (r - baseline) / (t + 1)

            # gradient bandit update
            for a2 in range(k):
                if a2 == a:
                    H[a2] += alpha * (r - baseline) * (1 - pi[a2])
                else:
                    H[a2] -= alpha * (r - baseline) * (pi[a2])

            avg_rewards[t] += r

    return avg_rewards / runs

grad_rewards = run_gradient_bandit(alpha=0.1)
plt.figure(figsize=(10,4))
plt.plot(grad_rewards, label="Gradient bandit (α=0.1)")
plt.xlabel("Steps"); plt.ylabel("Average reward")
plt.legend(); plt.grid(True); plt.show()


## Powiązanie z kontrolą MDP

Pomysły eksploracyjne z bandytów wracają w sterowaniu MDP (np. $\varepsilon$-greedy w SARSA / Q-learning).


### Notatki / wskazówki

- W testbedzie stacjonarnym średnia z próbek jest OK; w niestacjonarnym lepszy jest stały krok $\alpha$.
- Zwróć uwagę na rozstrzyganie remisów (`argmax`) — to może wpływać na wczesne zachowanie.
- UCB w praktyce wymaga sensownego traktowania akcji niepróbowanych (np. duży bonus).
